In [ ]:
import sys
import os
import yaml

In [ ]:
notebook_dir = os.getcwd()
src_dir = notebook_dir  # The notebook is already in src/
sys.path.insert(0, src_dir)

print(f"Added to path: {src_dir}")
print(f"Current working directory: {notebook_dir}")

In [ ]:
from app.trainer import OnPolicyTrainer, OnPolicySchedule

# Actor Critic

In [ ]:
from app.rl_agents import ActorCritic
from app.models import ValueModel, StochasticDiscretePolicy
from app.schedulers import ScheduleWrapper
from app.normalizer import Normalizer
from app.env_wrapper import GymnasiumWrapper
from app.buffer import RolloutBuffer
from app.renderer import Renderer
from app.rl_callbacks import WandbCallback
from app.logging_config import configure_logging

In [1]:
import numpy as np
np.__version__

'1.26.0'

In [ ]:
# Configure logger
app_logger = configure_logging()

In [ ]:
# Create Env
env = GymnasiumWrapper(
    cfg='CartPole-v1',
    num_envs=8,
    wrappers=[],
    render_mode=None,
    seed=42,
    obs_key=None,
    goal_key=None,
    ach_goal_key=None
    )

In [ ]:
# Create policy
policy = StochasticDiscretePolicy(
    env=env,
    layer_config=[
        {
            'type': 'dense',
            'params': {'units': 64, 'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        },
        {
            'type': 'dense',
            'params': {'units': 32, 'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        }
    ],
    output_config=[
        {
            'type': 'dense',
            'params': {'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        }
    ],
    optimizer_params={
        'type': 'Adam',
        'params': {'lr': 0.001}
    },
    lr_scheduler=None,
    distribution='categorical',
    device='cuda'
)

In [ ]:
# Create value model
value = ValueModel(
    env=env,
    layer_config=[
        {
            'type': 'dense',
            'params': {'units': 64, 'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        },
        {
            'type': 'dense',
            'params': {'units': 32, 'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        }
    ],
    output_config=[
        {
            'type': 'dense',
            'params': {'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        }
    ],
    optimizer_params={
        'type': 'Adam',
        'params': {'lr': 0.0001}
    },
    lr_scheduler=None,
    device='cuda'
)

In [ ]:
# Create Normalizers
state_normalizer = Normalizer(
    size=4,
    device='cuda'
)

advantage_normalizer = Normalizer(
    size=1,
    device='cuda'
)

In [ ]:
# Set params
discount = 0.99
policy_trace_decay = 0.0
value_trace_decay = 0.0
entropy_coefficient = 0.0
entropy_schedule = None
gae_coefficient = 0.95
save_dir = "E:/Documents/Programming/Projects/Reinforcement/PhoenX_RL/src/Trained_Models/Test_CartPole-v1_1/Reinforce/"
device = "cuda"

In [ ]:
# Build ActorCritic
agent = ActorCritic(
    policy=policy,
    value=value,
    discount=discount,
    policy_trace_decay=policy_trace_decay,
    value_trace_decay=value_trace_decay,
    entropy_coefficient=entropy_coefficient,
    # entropy_schedule=entropy_schedule,
    gae_coefficient=gae_coefficient,
    state_normalizer=state_normalizer,
    advantage_normalizer=advantage_normalizer,
    save_dir=save_dir,
    device=device
)


In [ ]:
# Create Buffer
buffer = RolloutBuffer(
    env=env,
    buffer_size=100,
    device=device
)

In [ ]:
# Create Schedule
schedule = OnPolicySchedule(
    unit='episode',
    num_units=1000,
    learn_unit='timestep',
    num_learn_units=800,
    seed=42
)


In [ ]:
# Create Renderer
renderer = Renderer(
    render_freq=1000,
    save_dir=save_dir,
    fps=30,
    codec='libx264'
)


In [ ]:
# Create callbacks
callbacks = [
    WandbCallback(
        project_name="CartPole-v1"
    )
]

In [ ]:
# Create trainer
trainer = OnPolicyTrainer(
    agent=agent,
    env=env,
    buffer=buffer,
    schedule=schedule,
    renderer=renderer,
    callbacks=callbacks
)

In [ ]:
trainer.train()

In [ ]:
trainer.buffer.states.shape

# Test agent.py

In [1]:
from scripts.agent import build_trainer_from_config_path as build

e:\Miniconda3\envs\rl_env\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment Reacher-v2 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
e:\Miniconda3\envs\rl_env\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment Pusher-v2 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
e:\Miniconda3\envs\rl_env\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment InvertedPendulum-v2 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
e:\Miniconda3\envs\rl_env\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment InvertedDoublePendulum-v2 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
e:\Miniconda3\envs\rl_env\Lib\site-packages\gymnasium\env

The Zen of Python, by Tim Peters

Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than *right* now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!


In [2]:
trainer = build("E:/Documents/Programming/Projects/Reinforcement/PhoenX_RL/src/Configs/actor_critic.yml")

In [3]:
trainer.renderer

Renderer(render_freq=1000, save_dir='E:/Documents/Programming/Projects/Reinforcement/PhoenX_RL/src/Trained_Models/CartPole-v1_test/ActorCritic/', fps=30, codec='libx264', logger=<Logger Renderer (WARNING)>)